## Part 1: Environment Setup

# 📚 Week 4: PDF Document Summarizer (RAG Project)

## 🎯 Session Overview

Welcome to Week 4! In this session, we'll build a complete **RAG (Retrieval-Augmented Generation)** application that can answer questions about PDF documents.

### What You'll Learn:
1. **PDF Processing**: Load and extract text from PDF files
2. **Text Chunking**: Split documents into manageable pieces
3. **Embeddings**: Convert text to numerical vectors
4. **Vector Stores**: Build efficient similarity search with FAISS
5. **RAG Pipeline**: Create a complete question-answering system
6. **LangChain Chains**: Use LCEL to build composable pipelines

### Why RAG Matters:
- **Reduces Hallucinations**: LLM answers are grounded in your documents
- **Private Data**: Query your own documents without fine-tuning
- **Up-to-date Info**: Works with the latest information
- **Cost Effective**: No need for expensive model training
- **Explainable**: Can trace answers back to source documents

### Real-World Applications:
- 📄 Legal document analysis
- 📚 Research paper summarization
- 🏢 Company knowledge bases
- 📋 Policy and compliance Q&A
- 📖 Educational content assistance

### Session Structure:
- **Part 1**: Environment Setup
- **Part 2**: Loading PDF Documents
- **Part 3**: Text Chunking
- **Part 4**: Creating Embeddings
- **Part 5**: Building FAISS Vector Store
- **Part 6**: Building the RAG Pipeline
- **Part 7**: Interactive Chat Interface

---

In [ ]:
# ============================================================
# INSTALLING REQUIRED PACKAGES
# ============================================================
# Install required packages for RAG (Retrieval-Augmented Generation) application
# Uncomment the line below to install if needed:
# !pip install langchain langchain-openai langchain-community pypdf faiss-cpu python-dotenv langsmith

# Package Overview:
# - langchain: Core framework for building LLM applications
# - langchain-openai: Integration with OpenAI models (GPT, embeddings)
# - langchain-community: Community-contributed integrations (PyPDFLoader, etc.)
# - pypdf: Library for reading and extracting text from PDF files
# - faiss-cpu: Facebook AI Similarity Search - vector database for fast similarity search
# - python-dotenv: Load environment variables from .env file (for API keys)
# - langsmith: LangChain's observability and debugging platform

In [ ]:
# ============================================================
# VERIFY LANGCHAIN INSTALLATION
# ============================================================
# Uncomment the following line to check your LangChain version:
# !pip show langchain

# This helps ensure the correct version is installed
# Recommended version: 0.1.0 or higher

In [ ]:
# ============================================================
# IMPORT NECESSARY LIBRARIES
# ============================================================

import os
from dotenv import load_dotenv

# LangChain Document Loaders
from langchain_community.document_loaders import PyPDFLoader  # For loading PDF files

# Text Splitting
from langchain_text_splitters import RecursiveCharacterTextSplitter  # Smart text chunking

# OpenAI Integration
from langchain_openai import OpenAIEmbeddings, ChatOpenAI  # Embeddings and LLM models

# Vector Store
from langchain_community.vectorstores import FAISS  # Vector database for semantic search

# Prompting and Output Parsing
from langchain_core.prompts import ChatPromptTemplate  # For creating structured prompts
from langchain_core.output_parsers import StrOutputParser  # Parse LLM output as string

# ============================================================
# LOAD ENVIRONMENT VARIABLES
# ============================================================
# Load API keys from .env file
load_dotenv()

# Set OpenAI API Key as environment variable
# Make sure you have a .env file with: OPENAI_API_KEY=your_api_key_here
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

print("✅ Libraries imported successfully!")

# ============================================================
# CONCEPT: What is RAG (Retrieval-Augmented Generation)?
# ============================================================
# RAG combines information retrieval with language generation:
# 1. RETRIEVE: Find relevant information from your documents
# 2. AUGMENT: Add that information to the LLM's context
# 3. GENERATE: LLM creates an answer based on retrieved context
#
# Benefits:
# - Reduces hallucinations (LLM making up facts)
# - Enables querying private/custom documents
# - More accurate and contextually relevant answers
# - No need to fine-tune the LLM on your data

d:\Mentoring\learwithsarvesh\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Libraries imported successfully!


### 🔑 Key Concepts Explained:

**1. Why do we need all these libraries?**
- Each library serves a specific purpose in the RAG pipeline
- Modular design allows flexibility and customization

**2. What is an API Key?**
- Authentication token that allows you to use OpenAI's services
- Should be kept secret (never share or commit to Git)
- Stored in `.env` file for security

**3. Environment Variables**
- Secure way to store sensitive information (API keys, passwords)
- `python-dotenv` loads them from `.env` file
- Prevents hardcoding secrets in your code

## Part 2: Loading PDF Documents

### Understanding PDF Loading
- We'll use PyPDFLoader to extract text from PDF files
- Each page becomes a separate document with metadata
- This allows us to track which page information came from

### Key Questions:
1. Why is metadata important when loading PDFs?
2. What information does metadata contain?
3. How would you handle PDFs with different formats or encryption?

In [ ]:
# ============================================================
# LOADING A PDF DOCUMENT
# ============================================================

# Step 1: Define the path to your PDF file
# Use absolute path or relative path to your PDF
pdf_path = "D:/Mentoring/learwithsarvesh/12weekcrash Course/Week4 - PDF Document Summarizer (RAG Project)/HDFC-Life-Group-Term-Life-Policy.pdf"

# Step 2: Create a PyPDFLoader instance
# PyPDFLoader is a specialized document loader for PDF files
loader = PyPDFLoader(pdf_path)

# Step 3: Load the documents
# The load() method extracts text from each page
# Each page becomes a separate Document object
documents = loader.load()

# Step 4: Print the number of pages loaded
print(f"📄 Loaded {len(documents)} pages from PDF")

# Step 5: Display first page preview and metadata
print(f"\n📖 First Page Preview:\n{documents[0].page_content[:300]}...")
print(f"\n📋 Metadata: {documents[0].metadata}")

# ============================================================
# CONCEPT: Understanding Document Structure
# ============================================================
# Each document object contains:
# 1. page_content: The actual text extracted from the page
# 2. metadata: Information about the document (source, page number, etc.)
#
# Why is this structure useful?
# - Allows us to track which page information came from
# - Enables citation and source tracking
# - Helps with debugging and verification
#
# Metadata typically includes:
# - source: Path to the original PDF file
# - page: Page number (0-indexed)
# - total_pages: Total number of pages in the document

Loaded 30 pages from pdf
 First Page preview: 
F&U dated 15th October 2022                  UIN-101N169V02  P a g e  | 0                        
 
 
 
 
 
   HDFC Life Group Term Life 
 
OF 
 
 
«OWNERNAME» 
 
 
 
 
 
  
Based on the Proposal and the declarations and 
any 
statement made or referred to therein, 
We will pay the Benefits mentione...


### 💡 Important Concepts:

**PyPDFLoader vs Other Loaders**
- `PyPDFLoader`: For PDF files specifically
- `TextLoader`: For plain text files
- `CSVLoader`: For CSV files
- `UnstructuredLoader`: For various document types

**Why Load by Page?**
- Each page is a separate "chunk" initially
- Makes it easier to cite sources ("Found on page 5")
- Allows parallel processing of large documents
- Better memory management for huge PDFs

**Common PDF Loading Challenges:**
- Scanned PDFs (images): Need OCR (Optical Character Recognition)
- Password-protected PDFs: Need decryption
- Complex layouts: Tables, multi-column text
- Non-English text: Encoding issues

## Part 3: Text Chunking

### Why Chunking?
- LLMs have token limits for context
- Embeddings work better with focused, manageable chunks
- Better retrieval accuracy with smaller, semantic units

### Chunking Strategy:
- **chunk_size**: Size of each text chunk (characters)
- **chunk_overlap**: Overlap between chunks to maintain context
- **RecursiveCharacterTextSplitter**: Splits intelligently at paragraph/sentence boundaries

### Experiment:
- What happens if chunk_size = 500? Too small?
- What happens if chunk_overlap = 0? Lost context?
- Try adjusting these values and observe the results

### ⚠️ Common Misconception:
**"Smaller chunks = better retrieval"** - Not always correct! 

Context matters more than size. Too small chunks lose context; too large chunks have too much irrelevant info.

In [ ]:
# ============================================================
# TEXT CHUNKING WITH RecursiveCharacterTextSplitter
# ============================================================

# Step 1: Initialize the text splitter with smart splitting strategy
text_splitter = RecursiveCharacterTextSplitter(
    # Separators define the hierarchy of splitting points
    separators=[
        "\n\n",  # First try to split at paragraph breaks (best for context)
        "\n",    # Then try newlines (sentence/line breaks)
        " ",     # Then try spaces (word breaks)
        ""       # Finally split character by character (last resort)
    ],
    chunk_size=1000,        # Maximum characters per chunk
    chunk_overlap=200,      # Overlap between chunks to preserve context
    length_function=len,    # Function to measure chunk length
)

# Step 2: Split the documents into chunks
# This processes all pages and creates smaller, manageable pieces
chunks = text_splitter.split_documents(documents)

# Step 3: Print number of chunks created
print(f"📊 Created {len(chunks)} chunks from {len(documents)} pages")

# Step 4: Display first chunk content and metadata
print(f"\n📄 Sample Chunk:\n{chunks[0].page_content}")
print(f"\n" + "="*50)
print(f"\n📋 Chunk Metadata: {chunks[0].metadata}")

# ============================================================
# CONCEPT: Why Recursive Character Text Splitter?
# ============================================================
# The "Recursive" approach tries to split at the most logical points:
# 1. Tries paragraphs first (best semantic boundaries)
# 2. Falls back to sentences if paragraphs too large
# 3. Falls back to words if sentences too large
# 4. Only splits mid-word as absolute last resort
#
# This preserves meaning better than arbitrary character splits!
#
# Key Parameters Explained:
# - chunk_size: Target size for each chunk (1000 chars is good for most cases)
# - chunk_overlap: Creates continuity between chunks (prevents cutting context)
# - separators: Defines the splitting hierarchy (most important to least)
#
# Why overlap?
# - Prevents losing context at chunk boundaries
# - If a key sentence is split between chunks, overlap ensures it appears somewhere complete
# - Example: 200 char overlap means last 200 chars of chunk N appear in chunk N+1

Created 115 chunks from 30 pages

 Sample Chunk: 
 F&U dated 15th October 2022                  UIN-101N169V02  P a g e  | 0                        
 
 
 
 
 
   HDFC Life Group Term Life 
 
OF 
 
 
«OWNERNAME» 
 
 
 
 
 
  
Based on the Proposal and the declarations and 
any 
statement made or referred to therein, 
We will pay the Benefits mentioned in this Policy 
subject to the terms and conditions contained 
herein 
 
 
 
 
 
 
<< Designation of the Authorised Signatory >>

 --------------------

 Chunk Metadata: {'producer': 'Microsoft® Office Word 2007', 'creator': 'Microsoft® Office Word 2007', 'creationdate': '2023-08-24T19:47:11+05:30', 'title': 'Exide Life Group Term Life (UIN 114N012V03) – Terms and Conditions', 'author': 'Atul Bhatia', 'moddate': '2023-08-24T19:47:11+05:30', 'source': 'D:/Mentoring/learwithsarvesh/12weekcrash Course/Week4 - PDF Document Summarizer (RAG Project)/HDFC-Life-Group-Term-Life-Policy.pdf', 'total_pages': 30, 'page': 0, 'page_label': '1'}


### 🎯 Chunking Strategy Guidelines:

**Choosing chunk_size:**
- **Small (200-500)**: Good for FAQ, short Q&A, precise retrieval
- **Medium (500-1000)**: Ideal for general documents, articles, blogs
- **Large (1000-2000)**: Better for legal docs, research papers, complex context

**Choosing chunk_overlap:**
- Typically 10-20% of chunk_size
- Too little: Risk losing context at boundaries
- Too much: Redundancy and slower processing

**Example Scenarios:**

| Document Type | chunk_size | chunk_overlap | Reason |
|--------------|------------|---------------|---------|
| FAQ Document | 300 | 50 | Short, focused answers |
| Blog Posts | 800 | 150 | Balanced context |
| Legal Contract | 1500 | 300 | Need complete clauses |
| Research Paper | 1200 | 200 | Complex concepts |

### 🚨 Token Limits Matter!

**Example Scenario:**
- LLM has 16k token context window
- You send 20k tokens of context → ❌ Error!
- LLM will refuse or truncate the input

**This is why chunking is critical for RAG systems!**

In [ ]:
# ============================================================
# ANALYZE CHUNK STATISTICS
# ============================================================

# Step 1: Calculate min, max, and average chunk sizes
# This helps us understand how the splitter performed
chunk_lengths = [len(chunk.page_content) for chunk in chunks]

print("📊 Chunk Statistics:")
print(f"   - Minimum chunk size: {min(chunk_lengths)} characters")
print(f"   - Maximum chunk size: {max(chunk_lengths)} characters")
print(f"   - Average chunk size: {sum(chunk_lengths) / len(chunk_lengths):.0f} characters")
print(f"\n{'='*60}")
print("✅ All chunks are within acceptable size range!")

# ============================================================
# WHY ANALYZE CHUNK STATS?
# ============================================================
# Checking statistics helps you verify:
# 1. No chunks exceed your target size (important for embeddings)
# 2. Distribution is relatively uniform (no huge outliers)
# 3. Splitter is working as expected
#
# If you see:
# - Max size >> chunk_size: Splitter couldn't find good breakpoints
# - Min size << average: You might have many tiny chunks (inefficient)
# - High variance: Consider adjusting separators or chunk_size

 Chunk Stats
 - Minimum chunk size : 232 characters
 - Maximum chunk size : 999 characters
 - Average chunk size : 849 characters

 ==============This is the Stats of the chunks==============


## Part 4: Creating Embeddings

### What are Embeddings?
- Vector representations of text that capture semantic meaning
- Similar texts have similar vectors (close in vector space)
- Enable semantic search (find meaning, not just keywords)

### OpenAI Embeddings:
- Model: `text-embedding-3-small` (1536 dimensions)
- Converts text → numerical vectors
- These vectors are stored in the vector database

### Think About:
1. What does "1536 dimensions" mean?
2. Why do similar sentences have similar embeddings?
3. How would cosine similarity help us find relevant chunks?

### 🏥 Real-World Analogy: Medical Health Reports

**Imagine a health report with 10 parameters:**
- BP, Blood Sugar, Cholesterol, HB, Vitamins, etc.
- Full report: 10,000 dimensions (too complex!)

**Embeddings are like a summary:**
- Convert 10 parameters → 5 key health indicators
- Reduces complexity while preserving essential information
- Easier to compare: "Is Patient A similar to Patient B?"

**Same concept with text:**
- Full text has thousands of words (high dimensional)
- Embeddings compress to 1536 dimensions
- Captures semantic meaning for comparison

### 🤖 OpenAI Model Options:

**LLM Models (for text generation):**
- `gpt-3.5-turbo` - Fast, cost-effective
- `gpt-4o-mini` - Balanced performance
- `gpt-4o` - Most capable, slower, more expensive

**Embedding Models (for converting text to vectors):**
- `text-embedding-3-small` - 1536 dimensions, faster, cheaper ⭐ (We're using this!)
- `text-embedding-3-large` - 3072 dimensions, higher quality, more expensive
- `text-embedding-ada-002` - Legacy model, 1536 dimensions

**Important:** Don't confuse LLM models with embedding models! They serve different purposes.

In [ ]:
# ============================================================
# INITIALIZE OPENAI EMBEDDINGS
# ============================================================

# Step 1: Create OpenAIEmbeddings instance
# Using text-embedding-3-small model (1536 dimensions)
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
    # Note: This is an EMBEDDING model, NOT a chat model like gpt-3.5-turbo
)

# Step 2: Test with a sample text
# This converts text into a vector representation
sample_text = "What is artificial intelligence?"
sample_embedding = embeddings.embed_query(sample_text)

# Step 3: Print embedding dimension and first 5 values
print(f"✅ Embeddings initialized successfully!")
print(f"📊 Embedding Dimension: {len(sample_embedding)}")
print(f"🔢 First 5 values: {sample_embedding[:5]}")
print(f"\n💡 Each chunk will be converted to a {len(sample_embedding)}-dimension vector")

# ============================================================
# CONCEPT: What Are Embeddings?
# ============================================================
# Embeddings convert text into numerical vectors that capture meaning:
# 
# Text: "The cat sat on the mat"
#   ↓
# Vector: [0.023, -0.145, 0.891, ..., 0.234]  (1536 numbers)
#
# Why are embeddings useful?
# 1. **Semantic Similarity**: Similar meanings = similar vectors
#    - "cat" and "feline" will have similar embeddings
#    - "cat" and "car" will have different embeddings
#
# 2. **Mathematical Operations**: Can calculate distance between meanings
#    - Cosine similarity measures how "close" two texts are
#    - Enables semantic search (meaning-based, not keyword-based)
#
# 3. **Dimension Reduction**: Captures essence of text in fixed size
#    - Any length text → same size vector (1536 dims)
#    - Efficient storage and comparison
#
# Each dimension represents some learned semantic feature
# (we don't know exactly what, but the model learned useful patterns)

 Embeddings initialized successfully
 Embedding Dimension: 1536
 First 5 values: [0.006162669975310564, -0.01451562624424696, -0.03586096316576004, 0.0057395463809370995, 0.021922778338193893]

 Each chunk will be converted to a 1536 - dimension vector


In [ ]:
# ============================================================
# COMPARING: text-embedding-3-large (Higher Dimension Model)
# ============================================================

# Step 1: Initialize with larger embedding model
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-large"  # 3072 dimensions instead of 1536
)

# Step 2: Test with the same sample text
sample_text = "What is artificial intelligence?"
sample_embedding = embeddings.embed_query(sample_text)

# Step 3: Compare dimensions with previous model
print(f"✅ Embeddings initialized successfully!")
print(f"📊 Embedding Dimension: {len(sample_embedding)}")
print(f"🔢 First 5 values: {sample_embedding[:5]}")
print(f"\n💡 Each chunk will be converted to a {len(sample_embedding)}-dimension vector")

# ============================================================
# TRADE-OFFS: Small vs Large Embedding Models
# ============================================================
# text-embedding-3-small (1536 dims):
#   ✅ Faster processing
#   ✅ Lower cost per query
#   ✅ Smaller storage requirements
#   ❌ Slightly lower accuracy
#
# text-embedding-3-large (3072 dims):
#   ✅ Higher accuracy
#   ✅ Better semantic understanding
#   ❌ Slower processing
#   ❌ Higher cost
#   ❌ More storage needed
#
# For most applications, text-embedding-3-small is sufficient!
# Use 3-large only if you need maximum accuracy and have the budget.

 Embeddings initialized successfully
 Embedding Dimension: 3072
 First 5 values: [-0.008916644379496574, -0.0029415732715278864, -0.017454100772738457, -0.001493767718784511, 0.02350960485637188]

 Each chunk will be converted to a 3072 - dimension vector


### Demonstrating Semantic Similarity

**Question:** What should happen when we compare "The cat sat on the mat" vs "A feline rested on the rug"?

In [ ]:
import numpy as np

# ============================================================
# DEMONSTRATING SEMANTIC SIMILARITY WITH COSINE SIMILARITY
# ============================================================

# Helper function to calculate cosine similarity
def cosine_similarity(vec1, vec2):
    """
    Calculate cosine similarity between two vectors.
    Returns a value between -1 and 1:
    - 1.0 = identical meaning
    - 0.0 = completely unrelated
    - -1.0 = opposite meaning
    """
    return np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))

# Test sentences
text1 = "The cat sat on the wall"
text2 = "Cat likes to chase mice"
text3 = "Python is a programming language"

# Convert all texts to embeddings
embed1 = embeddings.embed_query(text1)
embed2 = embeddings.embed_query(text2)
embed3 = embeddings.embed_query(text3)

# Calculate similarities
similarity_1_2 = cosine_similarity(embed1, embed2)  # Both about cats
similarity_1_3 = cosine_similarity(embed1, embed3)  # Cat vs Programming

# Display results
print("🔍 Semantic Similarity Demonstration:")
print(f"\n📝 Text 1: '{text1}'")
print(f"📝 Text 2: '{text2}'")
print(f"🎯 Similarity Score: {similarity_1_2:.4f}")
print(f"\n📝 Text 1: '{text1}'")
print(f"📝 Text 3: '{text3}'")
print(f"🎯 Similarity Score: {similarity_1_3:.4f}")

print(f"\n💡 Notice: Texts about similar topics have higher similarity scores!")

# ============================================================
# CONCEPT: Cosine Similarity Explained
# ============================================================
# Cosine similarity measures the angle between two vectors:
#
# Similar texts (small angle):     Different texts (large angle):
#        ↗ Vector1                       Vector1 →
#       ↗ Vector2                                ↓ Vector2
#    (close together)                    (far apart)
#    Score: ~0.8-1.0                     Score: ~0.0-0.3
#
# Why cosine instead of distance?
# - Focuses on direction, not magnitude
# - Works well for normalized vectors
# - Range is always -1 to 1 (easy to interpret)
#
# Typical score ranges:
# - 0.9-1.0: Nearly identical meaning
# - 0.7-0.9: Very similar (same topic)
# - 0.5-0.7: Somewhat related
# - 0.0-0.5: Different topics
# - Negative: Rare, indicates opposing concepts

Semantic Similarity

 The cat sat on the wall

 Cat likes to chase mice
Similarity: 0.4611
Similarity: 0.1144


### 🎯 Key Takeaway: Why This Matters for RAG

When you ask: *"What are the health benefits of exercise?"*

The system will:
1. Convert your question to an embedding
2. Calculate cosine similarity with all chunk embeddings
3. Retrieve chunks with highest similarity scores
4. These chunks likely contain relevant information about health and exercise!

**This is semantic search in action** - finding meaning, not just matching keywords!

## Cosine Similarity:
Cosine Similarity measures how similar two pieces of text are by checking whether their meanings point towards same direction

- It doen't care for exact words
- Cosine means angle between 2 vectors
- Different Direction --> diff meaning

## Part 5: Building FAISS Vector Store

### What is FAISS?
- **Facebook AI Similarity Search**
- Efficient library for similarity search
- Stores embeddings and finds nearest neighbors
- Perfect for RAG applications

### How it Works:
1. Convert all chunks to embeddings
2. Store in FAISS index
3. Given a query, find most similar chunks
4. Return relevant context to the LLM

### Discussion:
- Why is an index better than comparing every chunk?
- What happens when you have 1 million documents?

### 📚 Library Analogy for Understanding FAISS Index

**Without an Index (Naive Search):**
- Like searching for a book in a library with no catalog
- You must check EVERY book one by one
- With 1 million documents, you make 1 million comparisons!
- ❌ Extremely slow and inefficient

**With FAISS Index:**
- Like a library catalog system
- Organizes books by topic, author, genre
- Find what you need in seconds, not hours
- ✅ Fast and efficient even with millions of documents!

**The Magic:**
FAISS uses clever algorithms to cluster similar embeddings together, so it only needs to check a small subset to find the best matches.

In [ ]:
# ============================================================
# CREATE FAISS VECTOR STORE
# ============================================================

print("⏳ Creating vector store... This may take a moment...")

# Create FAISS vector store from documents and embeddings
# This process:
# 1. Converts each chunk to an embedding (1536-dim vector)
# 2. Builds an efficient index for similarity search
# 3. Stores both embeddings and original text
vectorstore = FAISS.from_documents(
    documents=chunks,      # Our text chunks
    embedding=embeddings   # OpenAI embedding model
)

print(f"✅ Vector store created successfully!")
print(f"📊 Stored {len(chunks)} chunk embeddings")
print(f"🔍 Ready for semantic search!")

# ============================================================
# CONCEPT: What Just Happened?
# ============================================================
# FAISS.from_documents() performed several steps:
#
# 1. EMBEDDING CREATION:
#    For each chunk → API call to OpenAI → get 1536-dim vector
#    Example: "Insurance policy terms" → [0.12, -0.45, 0.89, ...]
#
# 2. INDEX BUILDING:
#    FAISS organizes vectors into a searchable structure
#    Uses algorithms like:
#    - Clustering (group similar vectors)
#    - Quantization (compress vectors)
#    - Indexing (fast lookup tables)
#
# 3. STORAGE:
#    Saves both:
#    - The embedding vectors (for similarity search)
#    - The original text chunks (to return as results)
#    - Metadata (page numbers, sources, etc.)
#
# Now we can search millions of vectors in milliseconds!
#
# FAISS vs Other Vector Databases:
# - FAISS: In-memory, super fast, free, good for prototypes
# - Pinecone: Cloud-hosted, scalable, managed service
# - Chroma: Persistent, easy to use, good for small-medium apps
# - Weaviate: Production-grade, feature-rich, self-hosted or cloud

⏳ Creating vector store... This may take a moment...
Vector store created successfully
 Stored 115 chunk embedddings
 Ready for semantic search


### Testing the Vector Store - Similarity Search

In [ ]:
# ============================================================
# TEST SIMILARITY SEARCH
# ============================================================

# Step 1: Create a test query
test_query = "What are the main topics discussed in this document?"

# Step 2: Search for top 5 similar chunks
# similarity_search finds chunks with embeddings closest to query embedding
relevant_docs = vectorstore.similarity_search(test_query, k=5)

# Step 3: Display the results with metadata
print(f"🔍 Query: {test_query}")
print(f"📋 Found {len(relevant_docs)} relevant chunks:\n")

for i, doc in enumerate(relevant_docs, 1):
    print(f"{'='*60}")
    print(f"📄 Chunk {i} (Page {doc.metadata.get('page', 'N/A')}):")
    print(doc.page_content[:300] + "...")
    print()

# ============================================================
# CONCEPT: How Similarity Search Works
# ============================================================
# Step-by-step process:
#
# 1. QUERY EMBEDDING:
#    "What are main topics?" → [0.23, -0.67, 0.91, ...]
#
# 2. COMPARE WITH ALL CHUNKS:
#    Calculate cosine similarity between query and each chunk
#    Query vector vs Chunk1 vector → similarity score
#    Query vector vs Chunk2 vector → similarity score
#    ... and so on
#
# 3. RANK BY SIMILARITY:
#    Sort all chunks by their similarity scores
#    Highest scores = most relevant chunks
#
# 4. RETURN TOP K:
#    Return the top k=5 most similar chunks
#    These are your "retrieved" documents for RAG!
#
# The parameter k controls how many chunks to retrieve:
# - k=3: Fewer, more focused results (faster, less context)
# - k=5: Balanced approach (recommended for most cases)
# - k=10: More comprehensive but may include less relevant info

 Query: What are the main topics discussed in this document?
Found 5 relevent_chunks:

Chunk 1 (Page 7):
15. Benefits means the Benefit as mentioned in Part C of this Policy Document. 
 
16. Benefit Expiry Age means the Age in Years last birthday as mentioned in the Policy Schedule. 
 
17. Certificate of Insurance in respect of an Insured Member , means the Certificate of Insurance 
issued by the Compa...
Chunk 2 (Page 1):
purchased HDFC Life Insurance Policy:  
 
 Policy Schedule   :  Summary of key features of your HDFC Life Insurance Policy 
 Premium Receipt   :  Acknowledgement of the first Premium paid by you 
 Terms & Conditions  :  Detailed terms of your Policy contract with HDFC Life                        ...
Chunk 3 (Page 24):
(c) disputes over Premium paid or payable in terms of insurance Policy; 
(d) misrepresentation of Policy terms and conditions at any time in the Policy document or Policy contract; 
(e) legal construction of insurance policies in so far as the disput

In [ ]:
# ============================================================
# DISPLAY SIMILARITY SCORES
# ============================================================

# Use similarity_search_with_score to get confidence scores
# Lower scores = more similar (distance-based metric)
results_with_score = vectorstore.similarity_search_with_score(test_query, k=3)

print(f"📊 Similarity Scores (lower = more similar):\n")
for i, (doc, score) in enumerate(results_with_score, 1):
    print(f"📄 Chunk {i}: Score = {score:.4f} | Page {doc.metadata.get('page', 'N/A')}")
    print(f"   Preview: {doc.page_content[:100]}...")
    print()

print(f"\n💡 These scores help us understand retrieval quality!")
print(f"   - Scores < 0.5: Highly relevant")
print(f"   - Scores 0.5-1.0: Moderately relevant")
print(f"   - Scores > 1.0: Less relevant")

# ============================================================
# CONCEPT: Understanding Similarity Scores
# ============================================================
# FAISS returns L2 (Euclidean) distance by default:
# - Score represents distance between vectors in vector space
# - Lower score = vectors are closer = more similar
# - Score of 0 = identical vectors
#
# How to interpret scores:
# 1. RELATIVE COMPARISON: Compare scores within one query
#    - If Chunk A score = 0.3 and Chunk B score = 0.8
#    - Chunk A is more relevant than Chunk B
#
# 2. ABSOLUTE THRESHOLD: Set a cutoff for relevance
#    - Only use chunks with score < 0.7
#    - This prevents retrieving irrelevant information
#
# 3. QUALITY CHECK: All scores very high? Query might be off-topic
#    - Helps detect when user asks about content not in the document
#    - Can trigger "I don't know" response
#
# You can also use different distance metrics:
# - L2 (default): Euclidean distance
# - Cosine: Measures angle between vectors (normalized L2)
# - Inner Product: Dot product (good for normalized vectors)

 Similarity Score (lower = more similar): 

Chunk 1: Score = 1.4221 | page 7
Chunk 2: Score = 1.4344 | page 1
Chunk 3: Score = 1.4586 | page 24

 These scores help us understand retrievel qualities!


### Save and Load Vector Store (Optional)

In [ ]:
# ============================================================
# SAVE VECTOR STORE LOCALLY (OPTIONAL BUT RECOMMENDED)
# ============================================================

# Save the vector store to disk for reuse
# This avoids re-creating embeddings every time (saves time and API costs!)
vectorstore.save_local("pdf_vectorstore")

print("💾 Vector store saved to 'pdf_vectorstore' folder")
print("\n📂 You can now load it later with:")
print("   vectorstore = FAISS.load_local('pdf_vectorstore', embeddings)")

# ============================================================
# WHY SAVE THE VECTOR STORE?
# ============================================================
# Benefits:
# 1. **Save Money**: Creating embeddings costs API calls to OpenAI
#    - If you have 1000 chunks, that's 1000 API calls!
#    - Saving lets you reuse without re-paying
#
# 2. **Save Time**: Embedding creation takes time
#    - Loading from disk is instant
#    - Great for development and testing
#
# 3. **Persistence**: Keep your work between sessions
#    - Close notebook, reopen later → just load the store
#    - No need to re-process the PDF
#
# To load later:
# from langchain_community.vectorstores import FAISS
# vectorstore = FAISS.load_local("pdf_vectorstore", embeddings)
#
# Note: You still need the embeddings model to load!
# (To convert queries to vectors during search)

## Part 6: Building the RAG Pipeline

### RAG Architecture:
```
User Query → Retrieve Relevant Chunks → Feed to LLM → Generate Answer
```

### Components:
1. **Retriever**: Gets relevant chunks from vector store
2. **LLM**: Generates answer based on retrieved context
3. **Prompt**: Instructs LLM how to use the context
4. **Chain**: Connects everything together

### Challenge:
- What should you tell the LLM about the context?
- How can you prevent the LLM from making up answers?

In [ ]:
# ============================================================
# INITIALIZE LLM AND RETRIEVER
# ============================================================

# Step 1: Create ChatOpenAI instance
# This is the language model that will generate answers
llm = ChatOpenAI(
    model="gpt-3.5-turbo",  # Fast and cost-effective model
    temperature=0           # Deterministic output (no randomness)
)

# Step 2: Create a retriever from the vector store
# Retriever is responsible for finding relevant chunks
retriever = vectorstore.as_retriever(
    search_type="similarity",    # Use similarity-based search
    search_kwargs={'k': 3}       # Retrieve top 3 most relevant chunks
)

# Step 3: Print confirmation messages
print(f"✅ LLM and Retriever initialized!")
print(f"🤖 LLM Model: gpt-3.5-turbo")
print(f"🌡️  Temperature: 0 (deterministic responses)")
print(f"🔍 Retriever will fetch {retriever.search_kwargs['k']} relevant chunks per query")

# ============================================================
# CONCEPT: LLM Parameters Explained
# ============================================================
# Temperature (0 to 2):
# - temperature=0: Always picks most likely next word
#   ✅ Best for factual Q&A, consistency, reproducibility
#   ❌ Less creative, can be repetitive
#
# - temperature=0.7: Balanced creativity and coherence
#   ✅ Good for conversational AI, content generation
#   ❌ Less predictable, may vary between runs
#
# - temperature=1.5+: Very creative/random
#   ✅ Good for brainstorming, creative writing
#   ❌ Can produce nonsensical outputs
#
# For RAG applications, use temperature=0 or very low!
# We want accurate, consistent answers based on documents.
#
# ============================================================
# CONCEPT: What is a Retriever?
# ============================================================
# A Retriever is an interface that:
# 1. Takes a query (string)
# 2. Returns relevant documents (list of Document objects)
#
# Our retriever configuration:
# - search_type="similarity": Find chunks similar to query
# - k=3: Return 3 most relevant chunks
#
# Alternative search types:
# - "mmr" (Maximal Marginal Relevance): Diverse results
# - "similarity_score_threshold": Only above certain score
#
# Why k=3?
# - Too few (k=1): May miss important context
# - Too many (k=10): Include irrelevant info, exceed context window
# - k=3-5: Sweet spot for most applications

 LLM and Retriever initialized!
 Retriever will fetch 3 relevent chunks per query


### Create a Custom Prompt Template

**Your Turn:** Design a prompt that:
1. Uses the retrieved context
2. Instructs the model to say "I don't know" if context isn't helpful
3. Asks for concise but informative answers

In [ ]:
# ============================================================
# CREATE CUSTOM PROMPT TEMPLATE
# ============================================================

# Step 1: Design system prompt with clear instructions
system_prompt = """You are an assistant for question-answering tasks based on PDF documents.
Use the following pieces of retrieved context to answer the question.
If you don't know the answer based on the context, just say that you don't know.
Keep the answer concise but informative.

Context: {context}
"""

# Step 2: Create ChatPromptTemplate with system and human messages
prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),  # System message sets the behavior
    ("human", "{input}")        # Human message is the user's question
])

# Step 3: Print the prompt template for verification
print("📝 System Prompt:")
print(system_prompt)
print("\n✅ Prompt template created successfully!")

# ============================================================
# CONCEPT: Why Prompt Engineering Matters
# ============================================================
# The prompt is the MOST IMPORTANT part of RAG!
# It tells the LLM how to use the retrieved context.
#
# Key elements of a good RAG prompt:
#
# 1. **Role Definition**: "You are an assistant for Q&A tasks"
#    - Sets the LLM's behavior and tone
#    - Helps it understand its purpose
#
# 2. **Context Instruction**: "Use the following pieces of retrieved context"
#    - Explicitly tells LLM to use provided information
#    - Prevents making up answers from training data
#
# 3. **Fallback Behavior**: "If you don't know, say you don't know"
#    - Critical for preventing hallucinations!
#    - Builds trust with users
#
# 4. **Output Format**: "Keep answer concise but informative"
#    - Guides the response style
#    - Can specify: bullet points, paragraphs, word limits
#
# 5. **Context Placeholder**: {context}
#    - Will be filled with retrieved chunks
#    - LLM only sees what we retrieve, nothing else!
#
# ============================================================
# PROMPT VARIATIONS FOR DIFFERENT USE CASES
# ============================================================
# 
# For Technical Documentation:
# "You are a technical support assistant. Provide step-by-step
# instructions based on the documentation. Use code examples
# when available in the context."
#
# For Legal Documents:
# "You are a legal document assistant. Quote relevant sections
# verbatim. Never interpret or provide legal advice."
#
# For Customer Support:
# "You are a friendly customer support agent. Answer based on
# the knowledge base. If unsure, suggest contacting support."
#
# For Research Papers:
# "You are a research assistant. Cite specific sections and
# page numbers. Maintain academic tone."

System Prompt:
You are an assistant for question-answering tasks based on PDF documents.
Use the following pieces of retrieved context to answer the question.
If you don't know the answer based on the context, just say that you don't know.
Keep the answer concise but informative.

Context: {context}



### Build the Complete RAG Chain

In [ ]:
# ============================================================
# BUILD THE COMPLETE RAG CHAIN USING LCEL
# ============================================================

# Step 1: Create a format_docs function to combine documents
def format_docs(docs):
    """
    Combine multiple document chunks into a single string.
    Each chunk is separated by double newlines for readability.
    """
    return "\n\n".join(doc.page_content for doc in docs)

# Step 2: Build chain with retriever | prompt | llm | parser
# Using LangChain Expression Language (LCEL) - the | operator chains components
rag_chain = (
    {
        # This dictionary defines inputs to the prompt template
        "context": (lambda x: x["input"]) | retriever | format_docs,  # Get user input → retrieve docs → format
        "input": lambda x: x["input"]  # Pass through the original question
    }
    | prompt              # Fill the prompt template with context and input
    | llm                 # Send to the language model
    | StrOutputParser()   # Parse LLM output as a string
)

# Step 3: Print confirmation that chain is created
print("✅ RAG Chain created successfully!")
print("\n🔗 Chain Flow:")
print("   User Question → Retriever → Context Formatting → Prompt → LLM → Answer")

# ============================================================
# CONCEPT: Understanding the RAG Chain
# ============================================================
# Let's break down what happens when you ask a question:
#
# 1. USER INPUT: "What are the benefits?"
#    ↓
# 2. CONTEXT PATH:
#    - lambda x: x["input"] → extracts "What are the benefits?"
#    - | retriever → searches vector store, returns top 3 chunks
#    - | format_docs → combines chunks into one string
#    Result: "Chunk1\n\nChunk2\n\nChunk3"
#    ↓
# 3. PROMPT CREATION:
#    - Takes context and input
#    - Fills the template: system prompt + context + user question
#    Result: Full prompt ready for LLM
#    ↓
# 4. LLM PROCESSING:
#    - ChatOpenAI receives the prompt
#    - Generates answer based on context
#    Result: LLM response object
#    ↓
# 5. OUTPUT PARSING:
#    - StrOutputParser extracts the text content
#    Result: Clean string answer
#
# ============================================================
# LCEL (LangChain Expression Language) Explained
# ============================================================
# The | operator creates a pipeline:
# - Output of left side → Input of right side
# - Makes code readable and composable
# - Easy to modify and extend
#
# Why use LCEL instead of functions?
# ✅ Automatic tracing and debugging
# ✅ Parallel execution when possible
# ✅ Streaming support
# ✅ Async/await support
# ✅ Built-in error handling
#
# Example - these are equivalent:
# 
# Old way (functions):
# docs = retriever.invoke(question)
# context = format_docs(docs)
# filled_prompt = prompt.invoke({"context": context, "input": question})
# response = llm.invoke(filled_prompt)
# answer = response.content
#
# LCEL way (chain):
# answer = rag_chain.invoke({"input": question})
# 
# Much cleaner and more powerful!

Chain is created!


### 🎯 Visual RAG Chain Flow

```
┌─────────────────┐
│  User Question  │
└────────┬────────┘
         │
         ▼
┌─────────────────┐
│   Retriever     │  ← Searches vector store
└────────┬────────┘
         │ (returns 3 docs)
         ▼
┌─────────────────┐
│  Format Docs    │  ← Combines chunks
└────────┬────────┘
         │
         ▼
┌─────────────────┐
│  Prompt Template│  ← Fills context + question
└────────┬────────┘
         │
         ▼
┌─────────────────┐
│      LLM        │  ← Generates answer
└────────┬────────┘
         │
         ▼
┌─────────────────┐
│  Output Parser  │  ← Extracts text
└────────┬────────┘
         │
         ▼
┌─────────────────┐
│  Final Answer   │
└─────────────────┘
```

### 📊 Understanding the Lambda Functions

**What does this do?**
```python
"context": (lambda x: x["input"]) | retriever | format_docs
```

**Step-by-step breakdown:**
1. Takes the user's question from input dictionary
2. Sends it to the retriever → gets relevant document chunks
3. Formats them into a clean text string
4. This becomes the `{context}` in the prompt template!

### Test the RAG Chain

In [ ]:
# ============================================================
# TEST THE RAG CHAIN
# ============================================================

# Step 1: Create a question about the PDF
question = "What is this document about? Give me a brief summary"

# Step 2: Invoke the chain with the question
print("🤔 Processing your question...")
answer = rag_chain.invoke({"input": question})

# Step 3: Display the answer
print(f"\n❓ Question: {question}\n")
print(f"{'='*60}")
print(f"💬 Answer:\n{answer}")
print(f"{'='*60}")

# Step 4: Show the retrieved context separately (for verification)
retrieved_docs = retriever.invoke(question)
print(f"\n📚 Retrieved {len(retrieved_docs)} relevant chunks:\n")

for i, doc in enumerate(retrieved_docs, 1):
    print(f"\n{'─'*60}")
    print(f"📄 Chunk {i} (Page {doc.metadata.get('page', 'N/A')}):")
    print(f"{'─'*60}")
    print(doc.page_content[:200] + '...')

# ============================================================
# CONCEPT: Verifying RAG Quality
# ============================================================
# Always inspect the retrieved chunks to ensure:
#
# 1. **RELEVANCE**: Are the chunks actually related to the question?
#    - If not, your embeddings or query might need improvement
#    - Consider query rephrasing or better chunking strategy
#
# 2. **COVERAGE**: Do the chunks contain enough information?
#    - If answer is incomplete, try increasing k (more chunks)
#    - Or improve your chunking strategy
#
# 3. **ACCURACY**: Does the LLM answer match the context?
#    - If LLM hallucinates despite good context → improve prompt
#    - If context is wrong → improve retrieval
#
# 4. **SOURCE TRACKING**: Can you trace the answer to specific pages?
#    - Important for citations and verification
#    - Metadata helps users verify information
#
# Red Flags to Watch For:
# ❌ Answer contradicts the retrieved chunks
# ❌ Retrieved chunks are completely off-topic
# ❌ Answer includes information not in chunks (hallucination)
# ❌ Answer is vague despite specific context
#
# Good Signs:
# ✅ Answer directly references the retrieved context
# ✅ Retrieved chunks are highly relevant
# ✅ Answer admits "I don't know" when context is insufficient
# ✅ Answer is specific and grounded in the documents

Question: What is this document about? Give me a breif summary

Answer: 
This document is related to an HDFC Life Insurance Policy. It includes information such as the Policy Schedule, Premium Receipt, Terms & Conditions, Service Options, Deletion Details, Salary Updation Details, and Change in Policy-Member Details. It emphasizes the importance of carefully reviewing the information, keeping the Policy Bond safe for availing benefits, and providing necessary details in case of any changes or deletions. The document also mentions the Certificate of Insurance for insured members and provides instructions for communication in the case of an employer-employee relationship.


Retrieved 3 relevent_chunks

 --Chunk 1 (Page 1)---
purchased HDFC Life Insurance Policy:  
 
 Policy Schedule   :  Summary of key features of your HDFC Life Insurance Policy 
 Premium Receipt   :  Acknowledgement of the first Premium paid by you 
 ---

 --Chunk 2 (Page 26)---
above, else provide us the details sep

## Part 7: Interactive Chat Interface

### Build a "Chat with PDF" System
Now let's create an interactive chat interface where users can ask multiple questions!

In [ ]:
# ============================================================
# CREATE INTERACTIVE CHAT FUNCTION
# ============================================================

def chat_with_pdf(rag_chain, retriever):
    """
    Interactive chat interface for PDF Q&A.
    
    Args:
        rag_chain: The RAG chain for generating answers
        retriever: The retriever for fetching relevant chunks
    """
    print("="*60)
    print("📚 PDF Chat Assistant - Ask me anything about your document!")
    print("="*60)
    print("Type 'quit', 'exit', or 'q' to end the conversation")
    print("="*60)
    
    while True:
        # Get user input
        question = input("\n🧑 You: ").strip()
        
        # Check for exit commands
        if question.lower() in ['quit', 'exit', 'q', '']:
            print("\n👋 Thanks for chatting! Goodbye!")
            break
        
        # Get response from RAG chain
        print("\n🤖 Assistant: Thinking...", end='\r')
        
        try:
            # Invoke the RAG chain
            answer = rag_chain.invoke({"input": question})
            print(f"🤖 Assistant: {answer}\n")
            
            # Show source information
            context_docs = retriever.invoke(question)
            print(f"📖 Sources: Found in {len(context_docs)} relevant sections")
            
            # Optional: Show which pages were used
            pages = set(doc.metadata.get('page', 'N/A') for doc in context_docs)
            print(f"📄 Pages referenced: {', '.join(map(str, sorted(pages)))}")
            
        except Exception as e:
            print(f"❌ Error: {str(e)}\n")
            print("Please try asking your question differently.")

# ============================================================
# CONCEPT: Building User-Friendly Interfaces
# ============================================================
# Key elements of good chat interfaces:
#
# 1. **CLEAR INSTRUCTIONS**
#    - Tell users how to exit
#    - Show what the system can do
#    - Provide examples if needed
#
# 2. **FEEDBACK**
#    - Show "Thinking..." while processing
#    - Display source information
#    - Indicate success or errors clearly
#
# 3. **ERROR HANDLING**
#    - Catch exceptions gracefully
#    - Provide helpful error messages
#    - Don't crash on bad input
#
# 4. **SOURCE TRANSPARENCY**
#    - Show which parts of the document were used
#    - Display page numbers for verification
#    - Build trust with users
#
# 5. **EXIT OPTIONS**
#    - Multiple ways to exit (quit, exit, q)
#    - Empty input also exits
#    - Friendly goodbye message
#
# ============================================================
# ENHANCEMENT IDEAS
# ============================================================
# You can improve this function by:
# 
# 1. **Conversation History**
#    - Remember previous questions
#    - Allow follow-up questions
#    - "Tell me more about that"
#
# 2. **Better Source Display**
#    - Show snippets from retrieved chunks
#    - Highlight relevant sentences
#    - Add page numbers to quotes
#
# 3. **Input Validation**
#    - Check question length
#    - Detect spam or irrelevant queries
#    - Suggest better phrasings
#
# 4. **Metrics Tracking**
#    - Log all questions and answers
#    - Track retrieval quality
#    - Identify common questions
#
# 5. **Rich Formatting**
#    - Use colored text
#    - Format code blocks
#    - Display tables nicely

# ============================================================
# TO RUN THE CHAT INTERFACE:
# ============================================================
# Uncomment the line below:
# chat_with_pdf(rag_chain, retriever)

In [ ]:
# ============================================================
# START THE INTERACTIVE CHAT
# ============================================================
# This launches the chat interface - you can ask multiple questions!
# Type your questions and get answers based on your PDF document

chat_with_pdf(rag_chain, retriever)

# ============================================================
# TIPS FOR ASKING GOOD QUESTIONS
# ============================================================
# ✅ Be specific: "What are the coverage limits?" instead of "Tell me about it"
# ✅ One topic at a time: Better retrieval for focused questions
# ✅ Use document terminology: Match the language in your PDF
# ✅ Ask follow-ups: Build on previous answers
#
# ❌ Avoid overly broad questions: "Tell me everything"
# ❌ Don't ask about info not in the document
# ❌ Avoid multi-part questions in one query

PDF Chat Assistant - Ans me anything about your Document!
Type 'quit', 'exit', 'q' to end our conversation

 Thinking
Assistant: This document is a Policy Document related to an HDFC Life Insurance Policy. It includes information about benefits, benefit expiry age, certificate of insurance, notices by the company under the policy, the entire contract, and details on the purchased HDFC Life Insurance Policy such as the policy schedule, premium receipt, terms & conditions, and service options. It also provides guidance on the importance of keeping the policy bond safe for availing policy benefits and advises communication of insurance details to employees in case of an employer-employee relationship.

Source Found in 3 relevent section

 Thinking
Assistant: The toll-free number mentioned in the document is 1800 266 9777.

Source Found in 3 relevent section

 Thinking
Assistant: The HDFC Life Group Term Life Insurance policy is a non-linked, non-participating Group Life Insurance policy. 

### 📏 Chunk Size Recommendations by Use Case

**1. Small Chunks (300-500 characters):**
- ✅ FAQ documents
- ✅ Short Q&A format
- ✅ Precise, specific answers needed
- ❌ Risk: Losing broader context

**2. Medium Chunks (500-800 characters):**
- ✅ Blog posts and articles
- ✅ Documentation
- ✅ General knowledge bases
- ✅ **Most versatile option**

**3. Large Chunks (800-1500+ characters):**
- ✅ Legal documents (need complete clauses)
- ✅ Research papers (complex concepts)
- ✅ Technical specifications
- ❌ Risk: Too much irrelevant information

**Remember:** Experiment with your specific documents to find the optimal size!

### Alternative: Batch Q&A Testing

In [ ]:
# ============================================================
# BATCH Q&A TESTING (ALTERNATIVE TO INTERACTIVE CHAT)
# ============================================================
# This approach is useful for:
# - Testing your RAG system with predefined questions
# - Comparing answers across different configurations
# - Automated evaluation and quality checks

# Step 1: Create a list of test questions
test_questions = [
    "What is the main purpose of this document?",
    "What are the key terms and conditions?",
    "What are the coverage limits?",
    "What is the claim process?",
    "Are there any exclusions mentioned?",
]

# Step 2: Loop through each question and get answers
print("="*60)
print("📋 BATCH Q&A TEST")
print("="*60)

for i, question in enumerate(test_questions, 1):
    print(f"\n{'─'*60}")
    print(f"❓ Question {i}: {question}")
    print(f"{'─'*60}")
    
    try:
        # Get answer from RAG chain
        answer = rag_chain.invoke({"input": question})
        print(f"💬 Answer: {answer}")
        
        # Get source information
        docs = retriever.invoke(question)
        pages = set(doc.metadata.get('page', 'N/A') for doc in docs)
        print(f"📚 Sources: Pages {', '.join(map(str, sorted(pages)))}")
        
    except Exception as e:
        print(f"❌ Error: {str(e)}")

print(f"\n{'='*60}")
print("✅ Batch testing complete!")

# ============================================================
# WHY BATCH TESTING?
# ============================================================
# Advantages:
# 1. **Reproducibility**: Same questions every time
# 2. **Comparison**: Test different chunk sizes, models, prompts
# 3. **Evaluation**: Measure quality systematically
# 4. **Documentation**: Create examples for users
# 5. **Regression Testing**: Ensure changes don't break things
#
# Best Practices:
# - Include edge cases (ambiguous, out-of-scope questions)
# - Mix specific and broad questions
# - Test questions users will actually ask
# - Save results for comparison
# - Track which questions work well and which don't

## 🎓 Homework & Next Steps

Congratulations on completing the RAG fundamentals! Now it's time to level up your skills.

### 🏆 Challenges (Pick One or Try All!)

---

#### 1. **Multi-PDF Support** (⭐⭐ Medium Difficulty)

**Goal**: Load and query multiple PDFs simultaneously

**Requirements**:
- Load 3+ different PDF files
- Track which PDF each answer came from
- Display filename in the response
- Allow users to filter by specific PDFs

**Hints**:
```python
# Add filename to metadata
loader1 = PyPDFLoader("doc1.pdf")
docs1 = loader1.load()
for doc in docs1:
    doc.metadata["source_file"] = "doc1.pdf"
```

**Learning Outcomes**:
- Document management at scale
- Metadata enrichment
- Source tracking and attribution

---

#### 2. **Citation System** (⭐⭐⭐ Hard Difficulty)

**Goal**: Make the system cite specific pages/sections

**Requirements**:
- Answers include citations: "(See page 5)"
- Show relevant excerpts with page numbers
- Format citations properly
- Allow users to verify claims

**Hints**:
```python
# Modify prompt to include page numbers
system_prompt = """Use the context to answer, and cite sources.
Format: "According to page X, ..."
Context: {context}
"""
```

**Learning Outcomes**:
- Advanced prompt engineering
- Information verification
- Building trust with users

---

#### 3. **Conversational Memory** (⭐⭐⭐ Hard Difficulty)

**Goal**: Remember previous questions and handle follow-ups

**Requirements**:
- Maintain conversation history
- Handle questions like "Tell me more about that"
- Support context-aware follow-ups
- Clear history when starting new topic

**Hints**:
```python
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationalRetrievalChain

memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True
)
```

**Learning Outcomes**:
- Stateful conversations
- Memory management
- Context-aware responses

---

#### 4. **Streamlit UI Development** (⭐⭐ Medium Difficulty)

**Goal**: Build a web interface for your RAG system

**Requirements**:
- File upload functionality
- Chat interface with history
- Display source documents
- Adjustable parameters (chunk size, k, temperature)

**Hints**:
```python
import streamlit as st

uploaded_file = st.file_uploader("Upload PDF", type="pdf")
if uploaded_file:
    # Process the PDF
    st.chat_message("user").write(question)
    st.chat_message("assistant").write(answer)
```

**Learning Outcomes**:
- Full-stack development
- User experience design
- Deployment considerations

---

#### 5. **Semantic Chunking** (⭐⭐⭐⭐ Very Hard)

**Goal**: Split at semantic boundaries instead of fixed sizes

**Requirements**:
- Detect topic/paragraph changes
- Create variable-size chunks
- Maintain semantic coherence
- Compare quality with fixed-size chunks

**Hints**:
```python
from langchain.text_splitter import SpacyTextSplitter
# Or implement custom logic based on embeddings similarity
```

**Learning Outcomes**:
- Advanced NLP techniques
- Chunking strategy optimization
- Quality evaluation

---

#### 6. **Advanced Retrieval Techniques** (⭐⭐⭐ Hard)

**Goal**: Implement hybrid search (vector + keyword)

**Requirements**:
- Combine FAISS with BM25 keyword search
- Implement MMR (Maximal Marginal Relevance)
- Add re-ranking of results
- Compare different retrieval strategies

**Hints**:
```python
from langchain.retrievers import BM25Retriever, EnsembleRetriever

bm25_retriever = BM25Retriever.from_documents(chunks)
ensemble_retriever = EnsembleRetriever(
    retrievers=[vectorstore.as_retriever(), bm25_retriever],
    weights=[0.5, 0.5]
)
```

**Learning Outcomes**:
- Hybrid search strategies
- Retrieval evaluation
- Performance optimization

---

### 📚 Essential Resources

**LangChain Documentation**:
- [RAG Tutorial](https://python.langchain.com/docs/use_cases/question_answering/)
- [Text Splitters](https://python.langchain.com/docs/modules/data_connection/document_transformers/)
- [Vector Stores](https://python.langchain.com/docs/modules/data_connection/vectorstores/)

**FAISS**:
- [GitHub Repository](https://github.com/facebookresearch/faiss)
- [Getting Started Guide](https://github.com/facebookresearch/faiss/wiki/Getting-started)

**LangSmith** (Observability & Debugging):
- [LangSmith Platform](https://smith.langchain.com)
- [Tracing Guide](https://docs.smith.langchain.com/)

**Vector Databases**:
- [Pinecone](https://www.pinecone.io/) - Managed vector DB
- [Chroma](https://www.trychroma.com/) - Open-source vector DB
- [Weaviate](https://weaviate.io/) - Enterprise vector search

---

### 🔬 Evaluation & Testing

Learn to measure your RAG system quality:

**Key Metrics**:
1. **Retrieval Quality**: Are we finding the right chunks?
2. **Answer Quality**: Are answers accurate and helpful?
3. **Latency**: How fast is the system?
4. **Cost**: API usage and efficiency

**Tools**:
- LangSmith for tracing
- RAGAS for RAG evaluation
- Manual test suites

---

### 🚀 Next Session Preview: Advanced RAG Techniques

**What's Coming in Week 5**:
1. **RAG Optimization**:
   - Query expansion and rephrasing
   - Re-ranking strategies
   - Hypothetical document embeddings (HyDE)

2. **Advanced Embeddings**:
   - Fine-tuning embeddings for your domain
   - Multi-lingual embeddings
   - Cross-encoder re-ranking

3. **Hybrid Search**:
   - Combining vector and keyword search
   - BM25 + Dense retrieval
   - Fusion techniques

4. **Evaluation Metrics**:
   - RAGAS framework
   - Custom evaluation metrics
   - A/B testing strategies

5. **Production Considerations**:
   - Caching strategies
   - Rate limiting
   - Error handling
   - Monitoring and logging

---

### 💡 Pro Tips

1. **Start Simple**: Get basic RAG working before adding complexity
2. **Test Thoroughly**: Use batch testing to catch regressions
3. **Monitor Costs**: Track OpenAI API usage
4. **Iterate on Prompts**: The prompt is often more important than the model
5. **Save Vector Stores**: Don't re-embed every time
6. **Version Control**: Track changes to chunking, prompts, and models
7. **Get Feedback**: Have real users test your system

---

### 🤝 Community & Support

**Questions?**
- Review session recordings
- Check course Discord/Slack
- Consult LangChain documentation
- Experiment and iterate!

**Share Your Work**:
- Post screenshots of your application
- Share interesting findings
- Help fellow students
- Build in public!

---

---

## 🎉 Congratulations!

### You've Built a Complete RAG System! 🚀

**What You Accomplished Today**:
- ✅ Loaded and processed PDF documents
- ✅ Implemented intelligent text chunking
- ✅ Created semantic embeddings
- ✅ Built a FAISS vector store
- ✅ Constructed a complete RAG pipeline with LCEL
- ✅ Created an interactive chat interface

### Why This Matters:

The RAG pattern you learned today is the **foundation** for countless real-world AI applications:

- **Enterprise Knowledge Management**: Companies use this to query internal documents
- **Customer Support**: Automated Q&A based on product documentation
- **Legal Tech**: Contract analysis and compliance checking
- **Healthcare**: Medical literature search and clinical decision support
- **Education**: Personalized tutoring based on course materials

### This Foundation Scales To:
- 📈 Millions of documents
- 👥 Thousands of concurrent users
- 🏢 Mission-critical production applications
- 🌍 Multi-lingual global systems
- 🔒 Secure enterprise deployments

### Your Next Steps:

1. **Practice**: Try different PDFs, adjust parameters, experiment!
2. **Challenge Yourself**: Pick a homework challenge and build it
3. **Share**: Show what you've built to get feedback
4. **Iterate**: RAG systems improve with tuning and testing

---

### 🌟 Key Takeaways to Remember:

1. **RAG is about context**: Give the LLM the right information at the right time
2. **Chunking matters**: Good chunks = good retrieval = good answers
3. **Embeddings capture meaning**: Semantic search finds relevant content, not just keywords
4. **Prompt engineering is critical**: How you instruct the LLM determines quality
5. **Always verify sources**: Show users where answers come from
6. **Iterate and improve**: First version won't be perfect - that's okay!

---

### 📝 Quick Reference Card

**Load PDF**:
```python
loader = PyPDFLoader("file.pdf")
docs = loader.load()
```

**Chunk Text**:
```python
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.split_documents(docs)
```

**Create Embeddings**:
```python
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
```

**Build Vector Store**:
```python
vectorstore = FAISS.from_documents(chunks, embeddings)
```

**Create RAG Chain**:
```python
retriever = vectorstore.as_retriever(search_kwargs={'k': 3})
rag_chain = (
    {"context": retriever | format_docs, "input": lambda x: x["input"]}
    | prompt | llm | StrOutputParser()
)
```

**Query**:
```python
answer = rag_chain.invoke({"input": "Your question here"})
```

---

### 🙏 Thank You!

**Great work today!** You've taken a significant step toward building production-ready AI applications.

**Keep Learning. Keep Building. Keep Shipping!** 🚀

---

**See you in Week 5 for Advanced RAG Techniques!** 📚✨